# Green AI Decision Support Framework
## Sustainable PEFT Fine-Tuning — Beginner Research Notebook

**Goal:** build a reproducible evidence base and decision engine for choosing a PEFT method under accuracy, GPU-memory, time, energy, and carbon constraints.

This notebook is intentionally small enough for a university student to run on Kaggle/Colab. Start with a smoke test, then run measured experiments. Do not use synthetic numbers as empirical results.

In [ ]:
# CELL 1 — Install dependencies
# Kaggle: Internet = ON, Accelerator = GPU. A single T4/L4 is preferred.
!pip -q install -U transformers datasets peft accelerate bitsandbytes codecarbon pynvml scikit-learn pandas matplotlib


In [ ]:
# CELL 2 — Settings (EDIT ONLY THIS CELL FIRST)
RUN_MODE = 'smoke'          # 'smoke' first, then 'real'
RUN_FULL_FT = False         # keep False on a T4 unless you deliberately test a small model
MODELS = {
    'small': 'Qwen/Qwen2.5-0.5B',
    'medium': 'Qwen/Qwen2.5-1.5B',
}
METHODS = ['lora', 'qlora', 'lora_fa']
SEEDS = [42] if RUN_MODE == 'smoke' else [13, 42, 2024]
TRAIN_N = 256 if RUN_MODE == 'smoke' else 2000
EVAL_N = 128 if RUN_MODE == 'smoke' else 500
EPOCHS = 1
MAX_LENGTH = 128
BATCH_SIZE = 2
GRAD_ACCUM = 4
OUTPUT_DIR = 'green_ai_results'


In [ ]:
# CELL 3 — Imports and reproducibility
import os, gc, json, time, random, math
from pathlib import Path
import numpy as np, pandas as pd, torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    BitsAndBytesConfig, TrainingArguments, Trainer, set_seed)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score

OUT = Path(OUTPUT_DIR); OUT.mkdir(exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda': print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# CELL 4 — Dataset
def load_sst2(train_n, eval_n):
    ds = load_dataset('stanfordnlp/sst2')
    train = ds['train'].shuffle(seed=42).select(range(min(train_n, len(ds['train']))))
    val = ds['validation'].shuffle(seed=42).select(range(min(eval_n, len(ds['validation']))))
    return train, val

def tokenize(train, val, tok):
    def f(batch):
        x = tok(batch['sentence'], truncation=True, padding='max_length', max_length=MAX_LENGTH)
        x['labels'] = batch['label']; return x
    return train.map(f, batched=True, remove_columns=train.column_names), val.map(f, batched=True, remove_columns=val.column_names)


In [ ]:
# CELL 5 — GPU power monitor
try:
    import pynvml; pynvml.nvmlInit(); NVML_OK = True
except Exception as e:
    NVML_OK = False; print('NVML unavailable:', e)

class GPUPowerMonitor:
    def __enter__(self):
        self.samples=[]; self.running=True; self.start=time.time()
        if NVML_OK and DEVICE=='cuda':
            import threading
            def loop():
                h=pynvml.nvmlDeviceGetHandleByIndex(0)
                while self.running:
                    try: self.samples.append((time.time(), pynvml.nvmlDeviceGetPowerUsage(h)/1000))
                    except: pass
                    time.sleep(1)
            self.thread=threading.Thread(target=loop,daemon=True); self.thread.start()
        return self
    def __exit__(self,*args):
        self.running=False
        if hasattr(self,'thread'): self.thread.join(timeout=2)
        self.seconds=time.time()-self.start
        if len(self.samples)>=2:
            ts=np.array([x[0] for x in self.samples]); pw=np.array([x[1] for x in self.samples])
            self.energy_wh=float(np.trapz(pw,ts)/3600); self.measurement='nvml_gpu'
        else:
            self.energy_wh=np.nan; self.measurement='unavailable'


In [ ]:
# CELL 6 — Model builder
def build_model(model_id, method):
    tok=AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    kwargs={'num_labels':2, 'torch_dtype':torch.float16 if DEVICE=='cuda' else torch.float32}
    if method=='qlora':
        q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
        kwargs['quantization_config']=q
    model=AutoModelForSequenceClassification.from_pretrained(model_id,**kwargs)
    model.config.pad_token_id=tok.pad_token_id
    if method=='full_ft':
        pass
    else:
        if method=='qlora': model=prepare_model_for_kbit_training(model)
        cfg=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,target_modules=['q_proj','v_proj'],
                       bias='none',task_type=TaskType.SEQ_CLS,modules_to_save=['score'])
        model=get_peft_model(model,cfg)
        if method=='lora_fa':
            for n,p in model.named_parameters():
                if 'lora_A' in n: p.requires_grad=False
    return model,tok


In [ ]:
# CELL 7 — One measured experiment
def run_one(model_name, method, seed):
    set_seed(seed); train,val=load_sst2(TRAIN_N,EVAL_N)
    model,tok=build_model(model_name,method); train,val=tokenize(train,val,tok)
    model.to(DEVICE)
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    total=sum(p.numel() for p in model.parameters())
    args=TrainingArguments(output_dir=str(OUT/'tmp'),num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,learning_rate=2e-4,logging_steps=20,
        eval_strategy='epoch',save_strategy='no',report_to='none',fp16=(DEVICE=='cuda'),
        gradient_checkpointing=(method=='qlora'))
    trainer=Trainer(model=model,args=args,train_dataset=train,eval_dataset=val,
                    compute_metrics=lambda p:{'accuracy':accuracy_score(p.label_ids,np.argmax(p.predictions,axis=-1)),
                                              'f1':f1_score(p.label_ids,np.argmax(p.predictions,axis=-1))})
    if DEVICE=='cuda': torch.cuda.reset_peak_memory_stats()
    t0=time.time()
    with GPUPowerMonitor() as mon: trainer.train()
    elapsed=time.time()-t0
    metrics=trainer.evaluate(); peak=(torch.cuda.max_memory_allocated()/2**30) if DEVICE=='cuda' else np.nan
    row={'model':model_name,'method':method,'seed':seed,'accuracy':metrics.get('eval_accuracy',np.nan),
         'f1':metrics.get('eval_f1',np.nan),'time_sec':elapsed,'gpu_peak_gb':peak,
         'energy_wh':mon.energy_wh,'energy_measurement':mon.measurement,
         'trainable_params':trainable,'total_params':total}
    with open(OUT/f"{method}_{model_name.split('/')[-1]}_{seed}.json",'w') as f: json.dump(row,f,indent=2)
    del trainer,model; gc.collect();
    if DEVICE=='cuda': torch.cuda.empty_cache()
    return row


In [ ]:
# CELL 8 — Run the experiment matrix
rows=[]
active_methods=list(METHODS)
if RUN_FULL_FT: active_methods=['full_ft']+active_methods
for tier,model_name in MODELS.items():
    # Full FT is intentionally excluded by default because it can OOM on free GPUs.
    for method in active_methods:
        for seed in SEEDS:
            print('Running:',tier,method,seed)
            try: rows.append(run_one(model_name,method,seed))
            except RuntimeError as e:
                print('FAILED:',e)
                if 'out of memory' in str(e).lower() and DEVICE=='cuda': torch.cuda.empty_cache(); gc.collect()

results=pd.DataFrame(rows); results.to_csv(OUT/'measured_results.csv',index=False); results


In [ ]:
# CELL 9 — Carbon accounting
# This is a separate estimate because NVML GPU energy is not whole-system energy.
# CodeCarbon can be added for runs where you want a grid-intensity based carbon estimate.
from codecarbon import EmissionsTracker
print('Important: do not back-fill carbon for old runs. Re-run the experiment with the tracker if carbon is a primary result.')


In [ ]:
# CELL 10 — Pareto filtering
def pareto_front(df):
    # maximize accuracy; minimize memory, time, energy
    cols=['accuracy','gpu_peak_gb','time_sec','energy_wh']
    x=df[cols].astype(float).to_numpy(); keep=[]
    for i in range(len(x)):
        dominated=False
        for j in range(len(x)):
            if i==j: continue
            no_worse=(x[j,0]>=x[i,0] and np.all(x[j,1:]<=x[i,1:]))
            strictly=(x[j,0]>x[i,0] or np.any(x[j,1:]<x[i,1:]))
            if no_worse and strictly: dominated=True; break
        keep.append(not dominated)
    return df.assign(pareto=keep)
if len(results): print(pareto_front(results))


In [ ]:
# CELL 11 — Green Efficiency Index (GEI)
# GEI is a decision score, not a physical measurement.
def minmax(s, higher=True):
    s=pd.to_numeric(s,errors='coerce'); lo,hi=s.min(),s.max()
    if hi==lo: return pd.Series(1.0,index=s.index)
    z=(s-lo)/(hi-lo); return z if higher else 1-z

def add_gei(df, w=(0.40,0.20,0.20,0.20)):
    out=df.copy()
    out['score_accuracy']=minmax(out.accuracy,True)
    out['score_memory']=minmax(out.gpu_peak_gb,False)
    out['score_energy']=minmax(out.energy_wh,False)
    out['score_time']=minmax(out.time_sec,False)
    a,m,e,t=w; out['GEI']=a*out.score_accuracy+m*out.score_memory+e*out.score_energy+t*out.score_time
    return out.sort_values('GEI',ascending=False)
if len(results):
    ranked=add_gei(results); display(ranked[['model','method','accuracy','gpu_peak_gb','energy_wh','time_sec','GEI']])


In [ ]:
# CELL 12 — Green AI Decision Engine
def recommend(df, min_accuracy=0.85, max_memory_gb=None, max_energy_wh=None):
    d=df.copy()
    d=d[d.accuracy>=min_accuracy]
    if max_memory_gb is not None: d=d[d.gpu_peak_gb<=max_memory_gb]
    if max_energy_wh is not None: d=d[d.energy_wh<=max_energy_wh]
    if d.empty: return None, 'No measured method satisfies all constraints.'
    r=add_gei(d).iloc[0]
    return r, f"Recommend {r.method} for {r.model}; GEI={r.GEI:.3f}"

# Example user scenario — change these values after real runs.
if len(results):
    rec,msg=recommend(results,min_accuracy=0.85,max_memory_gb=12)
    print(msg)
    if rec is not None: print(rec.to_dict())


## How to interpret the framework

The **benchmark is evidence**, not the final contribution. The decision framework takes measured results and applies explicit user constraints. A recommendation is valid only when the required measurements exist. Never replace missing energy/carbon measurements with synthetic values in the paper.

**Main research claim:** the framework helps practitioners choose a PEFT strategy under explicit performance and sustainability constraints. It does not claim that one method is universally greenest.

## Beginner run checklist

1. Open this notebook in Kaggle.
2. Turn **Internet ON** and select **one GPU**.
3. Run Cell 1. If imports fail, restart the session and run again.
4. Leave `RUN_MODE = 'smoke'`.
5. Run Cells 2–8. Make sure at least one LoRA and one QLoRA run finishes.
6. Check `green_ai_results/measured_results.csv`.
7. Only then change to `RUN_MODE = 'real'`.
8. Run the real experiment with three seeds.
9. Save the raw JSON files and CSV; do not edit measured numbers manually.
10. Run the Pareto, GEI, and Decision Engine cells.
11. Record GPU model, software versions, dataset sizes, and exact configuration in the paper.

### Important research-integrity rules
- Do not call synthetic results experimental results.
- Do not claim 7B/13B results unless you actually run those models.
- Do not claim LISA results until its implementation has been validated against the original method.
- Report GPU energy separately from whole-system energy.
- A recommendation is conditional on the stated constraints and measured candidate set.